# Домашняя работа: квантизация и дистилляция LLM

В этой работе вы самостоятельно проведёте эксперименты с квантизацией и дистилляцией LLM и проверите, сколько качества при этом теряет модель. Задача, на которую обучаемся, — разбор запросов к голосовому ассистенту из датасета MASSIVE (ru-RU): по запросу модель возвращает JSON с intent и slots. Подробнее: https://huggingface.co/datasets/AmazonScience/massive

### Сперва настроим окружение и подготовим код для экспериментов

In [1]:
# Раскомментируйте в новом окружении, затем перезапустите kernel:
# %pip install "torch==2.8.0" "transformers==4.55.2" "datasets==4.1.1" \
#     "accelerate==1.10.1" "llmcompressor==0.8.1" "compressed-tensors==0.12.2" \
#     "numpy==2.3.3" "pandas==2.2.3" "safetensors==0.8.0" "huggingface-hub==0.36.2" tqdm "peft==0.17.1"

In [2]:
import gc
import json
import math
import random
import re
import shutil
import struct
from collections import Counter
from collections.abc import Sequence
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from importlib.metadata import version
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import (
    AutoConfig, AutoModelForCausalLM, AutoTokenizer,
    CompressedTensorsConfig, GenerationConfig, PreTrainedModel, PreTrainedTokenizerBase,
    Trainer, TrainingArguments, set_seed,
)
import os
from huggingface_hub import snapshot_download
from peft import LoraConfig, get_peft_model

In [3]:
SEED = 42
ARTIFACTS = Path("artifacts_compression_hw")
ARTIFACTS.mkdir(exist_ok=True)
set_seed(SEED)

def clean_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def ensure_empty_dir(path: str | Path) -> Path:
    path = Path(path)
    if path.exists() and any(path.iterdir()):
        raise FileExistsError(f"{path} уже существует и не пустой")
    path.mkdir(parents=True, exist_ok=True)
    return path

environment = {
    p: version(p) for p in (
        "torch", "transformers", "datasets", "llmcompressor",
        "compressed-tensors", "numpy", "safetensors", "peft", "accelerate",
    )
}
print(environment)
(ARTIFACTS / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

def snapshot(repo, revision, kind="model", patterns=None):
    hub = Path(os.environ.get("HW7_HUB", "/hub"))
    local = hub / (kind + "s--" + repo.replace("/", "--")) / "snapshots" / revision
    if local.is_dir():
        return str(local)
    return snapshot_download(repo, revision=revision, repo_type=kind, allow_patterns=patterns)

def save_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def save_jsonl(path, rows):
    path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows),
                    encoding="utf-8")

{'torch': '2.8.0', 'transformers': '4.55.2', 'datasets': '4.1.1', 'llmcompressor': '0.8.1', 'compressed-tensors': '0.12.2', 'numpy': '2.3.3', 'safetensors': '0.8.0', 'peft': '0.17.1', 'accelerate': '1.10.1'}

## Загрузка MASSIVE

Используем датасет MASSIVE. Все калибровочные примеры и данные для дистилляции берём только из train. Оцениваем на валидации (dev).

In [4]:
DATASET_ID = "AmazonScience/massive"
DATASET_REVISION = "ed58ac423a2f4121720918bf5301577edce4ffd3"  # refs/convert/parquet
LOCALE = "ru-RU"

dataset_url = Path(snapshot(DATASET_ID, DATASET_REVISION, "dataset", [f"{LOCALE}/*/*.parquet"])) / LOCALE
data_files = {
    "train": f"{dataset_url}/train/0000.parquet",
    "validation": f"{dataset_url}/validation/0000.parquet",
    "test": f"{dataset_url}/test/0000.parquet",
}
raw = load_dataset("parquet", data_files=data_files)
print(raw)

DatasetDict({
    train: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 11514
    })
    validation: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2033
    })
    test: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2974
    })
})

In [5]:
example_index = 0
example_raw = raw["train"][example_index]
print("Исходная запись MASSIVE:")
print(json.dumps(
    {key: example_raw[key] for key in ("id", "utt", "annot_utt", "intent")},
    ensure_ascii=False, indent=2,
))

Исходная запись MASSIVE:
{
  "id": "1",
  "utt": "разбуди меня в девять утра в пятницу",
  "annot_utt": "разбуди меня в [time : девять утра] в [date : пятницу]",
  "intent": 48
}

### Сконвертируем данные для калибровки

В MASSIVE текст запроса лежит в поле `utt`, intent записан числом, а slots отмечены прямо в тексте поля `annot_utt`, например `[time : семь утра]`. Модель должна по запросу вернуть JSON с названием intent и списком slots в виде пар «тип — значение». Поэтому преобразуем запись в `text` — исходный запрос — и `gold` — эталонный JSON-ответ. Такой формат удобнее для калибровки и подсчёта метрик.

In [6]:
SLOT_PATTERN = re.compile(r"\[([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]")

def extract_slots(annot_utt: str) -> list[list[str]]:
    """Достаёт slots из разметки MASSIVE вида `[time : семь утра]`.

    Args:
        annot_utt: размеченный запрос из поля `annot_utt`.

    Returns:
        Список пар `[тип, значение]` в порядке появления в запросе.
    """
    return [[m.group(1).strip(), m.group(2).strip()]
            for m in SLOT_PATTERN.finditer(annot_utt)]

def target_json(intent: str, slots: list[list[str]]) -> str:
    """Собирает эталонный ответ в компактный JSON.

    Args:
        intent: название intent.
        slots: список пар `[тип, значение]`.

    Returns:
        Строка `{"intent":...,"slots":[...]}` без пробелов между полями.
    """
    return json.dumps(
        {"intent": intent, "slots": slots},
        ensure_ascii=False, separators=(",", ":"),
    )

def prepare_massive(raw_splits: DatasetDict) -> DatasetDict:
    """Переводит сплиты MASSIVE в формат запрос - эталонный JSON.

    Args:
        raw_splits: исходные сплиты MASSIVE с полями `id`, `utt`, `intent`, `annot_utt`.

    Returns:
        Сплиты с полями `id`, `text` (запрос) и `gold` (эталонный JSON).
    """
    intent_feature = raw_splits["train"].features["intent"]

    def convert(row: dict[str, Any]) -> dict[str, str]:
        intent = intent_feature.int2str(row["intent"])
        slots = extract_slots(row["annot_utt"])
        return {
            "id": str(row["id"]), "text": row["utt"],
            "gold": target_json(intent, slots),
        }

    return DatasetDict({
        split: ds.map(convert, remove_columns=ds.column_names)
        for split, ds in raw_splits.items()
    })

In [7]:

data = prepare_massive(raw)
INTENTS = list(raw["train"].features["intent"].names)
SLOT_TYPES = sorted({
    kind for row in raw["train"]
    for kind, value in extract_slots(row["annot_utt"])
})
train_data = data["train"]
DEV_SAMPLES = 256
dev_data = data["validation"]
if DEV_SAMPLES is not None:
    dev_data = dev_data.shuffle(seed=SEED).select(range(min(DEV_SAMPLES, len(dev_data))))
print("Число классов:", len(INTENTS), "типов slots:", len(SLOT_TYPES))

Число классов: 60 типов slots: 54

In [8]:
example_prepared = train_data[example_index]
print("Запрос модели (text):")
print(example_prepared["text"])
print("\nЭталонный ответ (gold):")
print(json.dumps(json.loads(example_prepared["gold"]), ensure_ascii=False, indent=2))

Запрос модели (text):
разбуди меня в девять утра в пятницу

Эталонный ответ (gold):
{
  "intent": "alarm_set",
  "slots": [
    [
      "time",
      "девять утра"
    ],
    [
      "date",
      "пятницу"
    ]
  ]
}

### Генерация ответов
Всем моделям даём одинаковую инструкцию и одинаковые описания допустимых классов. Генерация детерминирована: greedy decoding, одинаковый лимит токенов.

In [9]:
BASE_MODEL_ID = "Qwen/Qwen3-8B" # можно пробовать модели того же семейства, но поменьше, чтобы считалось быстрее

# Значения используются как пример, при необходимости можно регулировать
MAX_PROMPT_TOKENS = 2048
MAX_NEW_TOKENS = 128
GENERATE_BATCH_SIZE = 16

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
BASELINE_PATH = snapshot(BASE_MODEL_ID, "b968826d9c46dd6066d109eabc6255188de91218")

Device: cuda:0

In [10]:
system_prompt = (
    'Определи intent и извлеки slots. Верни только JSON вида '
    '{"intent":"...","slots":[["тип","значение"],...]}. '
    'Значения slots копируй из запроса. Если slots нет, верни пустой список. '
    "Не добавляй объяснения.\nДопустимые intents: " + ", ".join(INTENTS)
    + "\nДопустимые типы slots: " + ", ".join(SLOT_TYPES)
)

def encode_prompt(tokenizer: PreTrainedTokenizerBase, text: str) -> list[int]:
    """Токенизирует запрос в chat-формате с общим system prompt.

    Args:
        tokenizer: токенизатор модели с chat template.
        text: запрос пользователя.

    Returns:
        Token ids промпта, заканчивающиеся началом ответа ассистента.
    """
    ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_prompt},
         {"role": "user", "content": text}],
        tokenize=True, add_generation_prompt=True, enable_thinking=False,
    )
    if len(ids) > MAX_PROMPT_TOKENS:
        raise ValueError(f"Prompt имеет {len(ids)} токенов. Увеличьте лимит, не обрезайте запрос.")
    return ids

def generation_config(tokenizer: PreTrainedTokenizerBase) -> GenerationConfig:
    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id
    return GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=pad_token_id,
        use_cache=True,
    )

def trim_answer(token_ids: list[int], eos_token_id: int, pad_token_id: int) -> list[int]:
    """Отрезает паддинг, после генерации

    Returns:
        Токены ответа; eos остаётся последним, если модель его сгенерировала.
    """
    if eos_token_id in token_ids:
        return token_ids[:token_ids.index(eos_token_id) + 1]
    end = len(token_ids)
    while end and token_ids[end - 1] == pad_token_id:
        end -= 1
    return token_ids[:end]

@torch.inference_mode()
def generate_answer_ids(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    prompts: Sequence[list[int]],
) -> list[list[int]]:
    """Жадно генерирует ответы на батч промптов.

    Args:
        model: causal LM.
        tokenizer: токенизатор модели.
        prompts: token ids промптов; паддинг добавляется слева.

    Returns:
        Токены ответа для каждого промпта в том же порядке, без промпта и паддинга.
    """
    config = generation_config(tokenizer)
    width = max(len(ids) for ids in prompts)
    input_ids, attention_mask = [], []

    for ids in prompts:
        pad = width - len(ids)
        input_ids.append([config.pad_token_id] * pad + ids)
        attention_mask.append([0] * pad + [1] * len(ids))
    input_ids = torch.tensor(input_ids, device=model.device)
    attention_mask = torch.tensor(attention_mask, device=model.device)

    output = model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        generation_config=config, use_model_defaults=False, do_sample=False,
    )
    return [
        trim_answer(row, config.eos_token_id, config.pad_token_id)
        for row in output[:, input_ids.shape[1]:].tolist()
    ]

In [11]:
tokenizer = AutoTokenizer.from_pretrained(BASELINE_PATH)
print(BASE_MODEL_ID, len(encode_prompt(tokenizer, train_data[0]["text"])))

Qwen/Qwen3-8B 504

## Метрика качества

$$
Q = \tfrac{1}{2} \operatorname{IntentAccuracy} + \tfrac{1}{2} \operatorname{SlotMicroF1}.
$$

Slots сравниваются как **мультимножество** пар `(тип, значение)`: порядок не влияет, лишние дубликаты дают false positives. Неверный JSON или нарушение схемы считается ошибкой всего ответа.

Дополнительно считаем `frame_exact_match` и `valid_json_rate`. Если во всей выборке нет ни одного gold/predicted slot, slot F1 равен 1 только при валидности всех ответов, иначе 0.

In [12]:
def normalize_value(value: str) -> str:
    """Приводит значение slot к виду для сравнения."""
    return " ".join(value.casefold().split())

def parse_answer(
    text: str, intents: Sequence[str], slot_types: Sequence[str],
) -> dict[str, Any] | None:
    """Разбирает ответ модели и проверяет формат.

    Args:
        text: сырой ответ модели.
        intents: допустимые названия intent.
        slot_types: допустимые типы slots.

    Returns:
        Словарь с ключами `intent` и `slots` или None, если ответ невалиден.
    """
    try:
        obj = json.loads(text.strip())
        if not isinstance(obj, dict) or set(obj) != {"intent", "slots"}:
            return None
        if not isinstance(obj["intent"], str) or obj["intent"] not in intents:
            return None
        if not isinstance(obj["slots"], list):
            return None
        for slot in obj["slots"]:
            if not isinstance(slot, list) or len(slot) != 2:
                return None
            kind, value = slot
            if (not isinstance(kind, str) or kind not in slot_types
                    or not isinstance(value, str) or not value.strip()):
                return None
        return obj
    except (ValueError, TypeError):
        return None

def slot_counter(obj: dict[str, Any]) -> Counter:
    """Превращает slots ответа в мультимножество

    Args:
        obj: разобранный ответ с ключом `slots`.

    Returns:
        Counter пар `(тип, нормализованное значение)`.
    """
    return Counter((kind, normalize_value(value)) for kind, value in obj["slots"])

def compute_metrics(
    predictions: Sequence[str],
    golds: Sequence[str],
    intents: Sequence[str],
    slot_types: Sequence[str],
) -> dict[str, float]:
    """Считает метрики качества по ответам модели.

    Args:
        predictions: ответы модели.
        golds: эталонные JSON в том же порядке.
        intents: допустимые названия intent.
        slot_types: допустимые типы slots.

    Returns:
        Словарь с `quality`, `intent_accuracy`, `slot_micro_f1`,
        `frame_exact_match`, `valid_json_rate` и числом примеров `n`.
    """
    if len(predictions) != len(golds) or not golds:
        raise ValueError("Нужны непустые списки одинаковой длины.")
    correct_intent = exact = valid = tp = fp = fn = 0
    for text, gold_text in zip(predictions, golds):
        gold = parse_answer(gold_text, intents, slot_types)
        if gold is None:
            raise ValueError("Невалидная gold-разметка")
        pred = parse_answer(text, intents, slot_types)
        gold_slots = slot_counter(gold)
        pred_slots = Counter() if pred is None else slot_counter(pred)
        tp += sum((pred_slots & gold_slots).values())
        fp += sum((pred_slots - gold_slots).values())
        fn += sum((gold_slots - pred_slots).values())
        if pred is not None:
            valid += 1
            intent_ok = pred["intent"] == gold["intent"]
            correct_intent += int(intent_ok)
            exact += int(intent_ok and pred_slots == gold_slots)
    n = len(golds)
    denom = 2 * tp + fp + fn
    slot_f1 = 2 * tp / denom if denom else float(valid == n)
    intent_acc = correct_intent / n
    return {
        "quality": 0.5 * (intent_acc + slot_f1),
        "intent_accuracy": intent_acc,
        "slot_micro_f1": slot_f1,
        "frame_exact_match": exact / n,
        "valid_json_rate": valid / n,
        "n": n,
    }

def evaluate(
    model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase, ds: Dataset,
) -> tuple[dict[str, float], list[str]]:
    """Генерирует ответы на выборку батчами и считает метрики.

    Args:
        model: оцениваемая модель.
        tokenizer: её токенизатор.
        ds: выборка с полями `text` и `gold`.

    Returns:
        Метрики из compute_metrics и список ответов модели в порядке выборки.
    """
    model.eval()
    texts = ds["text"]
    predictions = []

    for start in tqdm(range(0, len(texts), GENERATE_BATCH_SIZE)):
        prompts = [encode_prompt(tokenizer, t) for t in texts[start:start + GENERATE_BATCH_SIZE]]
        answers = generate_answer_ids(model, tokenizer, prompts)
        predictions.extend(tokenizer.batch_decode(answers, skip_special_tokens=True))

    metrics = compute_metrics(predictions, ds["gold"], INTENTS, SLOT_TYPES)
    return metrics, predictions

def evaluate_checkpoint(
    model_path: str | Path, ds: Dataset,
) -> tuple[dict[str, float], list[str]]:
    """Загружает сохранённый checkpoint, оценивает его и выгружает из памяти.

    Args:
        model_path: каталог dense или квантизированного checkpoint.
        ds: выборка с полями `text` и `gold`.

    Returns:
        Метрики и ответы модели, как в evaluate.
    """
    model_path = str(model_path)
    if (Path(model_path) / "adapter_config.json").exists():
        from peft import PeftModel
        base = AutoModelForCausalLM.from_pretrained(BASELINE_PATH, torch_dtype=torch.bfloat16,
                                                   device_map=DEVICE)
        model = PeftModel.from_pretrained(base, model_path)
        try:
            return evaluate(model, AutoTokenizer.from_pretrained(BASELINE_PATH), ds)
        finally:
            del model, base
            clean_memory()
    config = AutoConfig.from_pretrained(model_path)
    quantization_kwargs = {}
    if getattr(config, "quantization_config", None):
        # Веса распаковываются в bf16 на 4-битной сетке
        quantization_kwargs["quantization_config"] = CompressedTensorsConfig(
            run_compressed=False
        )
    loader = AutoModelForCausalLM
    if (getattr(config, "quantization_config", {}) or {}).get("format") == "int-quantized":
        from transformers import Qwen3ForCausalLM
        # В этом формате INT8-тензор загружается в weight до распаковки в BF16.
        class FrozenQwen3(Qwen3ForCausalLM):
            def __init__(self, config):
                super().__init__(config)
                self.requires_grad_(False)
        loader = FrozenQwen3
    model = loader.from_pretrained(
        model_path, torch_dtype=torch.bfloat16,
        device_map=DEVICE,
        **quantization_kwargs,
    )
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        return evaluate(model, tokenizer, ds)
    finally:
        del model
        clean_memory()

def save_dev(name, metrics, predictions):
    rows = [dict(row, prediction=pred) for row, pred in zip(dev_data, predictions)]
    save_jsonl(ARTIFACTS / f"{name}_predictions.jsonl", rows)
    save_json(ARTIFACTS / f"{name}_dev.json", metrics)
    print(name, metrics)

Посчитаем метрики бейзлайна.

In [13]:
baseline_metrics, baseline_predictions = evaluate_checkpoint(BASELINE_PATH, dev_data)
save_dev("baseline", baseline_metrics, baseline_predictions)

baseline {'quality': 0.2781112132352941, 'intent_accuracy': 0.36328125, 'slot_micro_f1': 0.19294117647058823, 'frame_exact_match': 0.109375, 'valid_json_rate': 0.51171875, 'n': 256}

## Квантизация (5 баллов)

Используя код с семинара выполните следующие задания:
1) На примере одного из методов откалибруйте бейзлайн на MASSIVE train и на WikiText, сравните качество на dev. Есть ли влияние калибровочного датасета на метрику? (1 балл)

2) Поэкспериментируйте с разными конфигурациями методов AWQ, GPTQ, SmoothQuant (3 балла)

    Запустите минимум три конфигурации, пробуйте менять разные факторы:

    - битность
    - число калибровочных примеров;
    - group size
    - symmetric / asymmetric quantization;
    - smoothing_strength
    - Дополнительно (+1 балл) — какие слои оставить в исходной точности

3) Объясните результат — какие параметры наиболее значимы? (1 балл)


In [14]:
from llmcompressor import oneshot, reset_session
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier

CALIBRATION_SAMPLES = 64
QUANT_RUNS = [
    ("gptq_massive", "gptq", "massive", None),
    ("gptq_wiki", "gptq", "wiki", None),
    ("gptq_first_bf16", "gptq", "massive", 0),
    ("gptq_last_bf16", "gptq", "massive", 35),
    ("awq_massive", "awq", "massive", None),
    ("smoothquant_massive", "smoothquant", "massive", None),
]

def calibration(corpus):
    if corpus == "massive":
        ds = train_data.select_columns(["text"])
    else:
        pattern = "wikitext-2-raw-v1/train-00000-of-00001.parquet"
        root = snapshot("Salesforce/wikitext", "b08601e04326c79dfdd32d625aee71d232d685c3",
                        "dataset", [pattern])
        ds = load_dataset("parquet", data_files=str(Path(root) / pattern), split="train")
    return ds.filter(lambda row: bool(row["text"].strip())).shuffle(seed=SEED).select(range(CALIBRATION_SAMPLES))

def quantize(name, method, corpus, bf16_block):
    path = ARTIFACTS / name
    ignore = ["lm_head"]
    if bf16_block is not None:
        ignore.append(rf"re:model\.layers\.{bf16_block}\..*")
    if method == "smoothquant":
        recipe = [SmoothQuantModifier(smoothing_strength=0.5),
                  GPTQModifier(targets="Linear", scheme="W8A8", ignore=ignore)]
    else:
        modifier = AWQModifier if method == "awq" else GPTQModifier
        recipe = [modifier(targets="Linear", scheme="W4A16", ignore=ignore)]
    model = AutoModelForCausalLM.from_pretrained(BASELINE_PATH, torch_dtype=torch.bfloat16,
                                                 device_map=DEVICE)
    ds = calibration(corpus)
    oneshot(model=model, tokenizer=tokenizer, dataset=ds, recipe=recipe, max_seq_length=256,
            num_calibration_samples=len(ds), shuffle_calibration_samples=False)
    model.save_pretrained(path, save_compressed=True)
    tokenizer.save_pretrained(path)
    reset_session()
    del model
    clean_memory()
    metrics, predictions = evaluate_checkpoint(path, dev_data)
    save_dev(name, metrics, predictions)

for run in QUANT_RUNS:
    quantize(*run)

gptq_massive {'quality': 0.23076195520581114, 'intent_accuracy': 0.296875, 'slot_micro_f1': 0.16464891041162227, 'frame_exact_match': 0.07421875, 'valid_json_rate': 0.4140625, 'n': 256}
gptq_wiki {'quality': 0.32544432511086474, 'intent_accuracy': 0.43359375, 'slot_micro_f1': 0.21729490022172948, 'frame_exact_match': 0.08984375, 'valid_json_rate': 0.5859375, 'n': 256}
gptq_first_bf16 {'quality': 0.20471222310126583, 'intent_accuracy': 0.29296875, 'slot_micro_f1': 0.11645569620253164, 'frame_exact_match': 0.0859375, 'valid_json_rate': 0.4296875, 'n': 256}
awq_massive {'quality': 0.3865463051823417, 'intent_accuracy': 0.53125, 'slot_micro_f1': 0.2418426103646833, 'frame_exact_match': 0.08984375, 'valid_json_rate': 0.73828125, 'n': 256}
smoothquant_massive {'quality': 0.3672937369246862, 'intent_accuracy': 0.49609375, 'slot_micro_f1': 0.2384937238493724, 'frame_exact_match': 0.12890625, 'valid_json_rate': 0.69140625, 'n': 256}

gptq_last_bf16 {'quality': 0.2151501225490196, 'intent_accuracy': 0.2734375, 'slot_micro_f1': 0.1568627450980392, 'frame_exact_match': 0.0625, 'valid_json_rate': 0.3828125, 'n': 256}

### Выводы по квантизации

Больше всего на качество повлиял выбор метода. На тех же 64 запросах MASSIVE и при одинаковом размере весов AWQ опередил GPTQ примерно на 16 процентных пунктов по Q. Для этой задачи я бы выбрал AWQ.

Результат калибровки GPTQ оказался неожиданным. Я ожидал, что лучше подойдут русские запросы из MASSIVE, ведь на них мы проверяли качество. Но с английскими текстами Википедии Q получилась выше — 0,33 против 0,23. Я взял по 64 примера из каждого корпуса. В среднем запрос MASSIVE содержал 13 токенов, а отрывок WikiText — 91. Возможно, WikiText дал лучший результат потому, что для калибровки было примерно в семь раз больше токенов. При одинаковом числе токенов я эти корпуса не сравнивал.

Оставлять первый или последний блок в BF16 не помогло. В обоих случаях веса выросли с 6,1 до 6,4 ГБ. С первым блоком в BF16 качество упало примерно на 3 процентных пункта, с последним — на 2.

Для SmoothQuant я взял конфигурацию W8A8 из семинара. Она дала Q около 0,37 при размере весов 9,4 ГБ. AWQ показал чуть лучшее качество, около 0,39, и занял 6,1 ГБ.

## Hard-label дистилляция (5 баллов)

Теперь реализуем hard-label дистилляцию знаний от учителя **Qwen3-32B** в бейзлайн.

### Задания
1) Измерьте качество учителя на dev и сравните с бейзлайном: насколько учитель сильнее? (1 балл)
2) Получите жадные ответы учителя на сэмпле из train. Используйте батч для ускорения при необходимости (1 балл)
3) Реализуйте код для обучения ученика на ответах учителя (1 балл)
4) Проведите эксперименты с обучением, получите модель ученика, отклоняющуюся на dev не более чем на 10% от метрики учителя (2 балла)


### Измерим качество учителя и сравним с бейзлайном (1 балл)

In [15]:
STUDENT_MODEL_ID = BASE_MODEL_ID
TEACHER_MODEL_ID = "Qwen/Qwen3-32B"
TEACHER_PATH = snapshot(TEACHER_MODEL_ID, "9216db5781bf21249d130ec9da846c4624c16137")
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_PATH)
assert teacher_tokenizer.get_vocab() == tokenizer.get_vocab()
assert teacher_tokenizer.chat_template == tokenizer.chat_template
assert teacher_tokenizer.eos_token_id == tokenizer.eos_token_id
assert teacher_tokenizer.pad_token_id == tokenizer.pad_token_id

In [16]:
teacher = AutoModelForCausalLM.from_pretrained(TEACHER_PATH, torch_dtype=torch.bfloat16,
                                              device_map=DEVICE).eval()

In [17]:
teacher_metrics, teacher_predictions = evaluate(teacher, teacher_tokenizer, dev_data)
save_dev("teacher", teacher_metrics, teacher_predictions)

teacher {'quality': 0.5784188329646018, 'intent_accuracy': 0.78515625, 'slot_micro_f1': 0.37168141592920356, 'frame_exact_match': 0.31640625, 'valid_json_rate': 0.9765625, 'n': 256}

### Получим ответы учителя на трейне (1 балл)

Не забывайте выгружать модели и очищать память.

In [18]:
DISTILL_SAMPLES = 384

def collect_teacher_answers(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    rows: Dataset,
    output_path: str | Path,
) -> Path:
    """Генерируем ответы учителя батчами и сохраняем их в answers.jsonl.

    Args:
        model: модель-учитель.
        tokenizer: общий токенизатор учителя и ученика.
        rows: примеры train с полями `id`, `text`, `gold`.
        output_path: пустой каталог для ответов.

    Returns:
        Каталог с answers.jsonl; в каждой строке `id`, `text`, `gold`,
        `teacher_answer`, `prompt_ids`, `answer_ids`.
    """
    output_path = Path(output_path)
    output_path.mkdir(exist_ok=True)
    records = []
    row_list = list(rows)

    for start in tqdm(range(0, len(row_list), GENERATE_BATCH_SIZE)):
        batch = row_list[start:start + GENERATE_BATCH_SIZE]
        prompts = [encode_prompt(tokenizer, row["text"]) for row in batch]
        answers = generate_answer_ids(model, tokenizer, prompts)
        for row, prompt, answer in zip(batch, prompts, answers):
            records.append(dict(row, prompt_ids=prompt, answer_ids=answer,
                                teacher_answer=tokenizer.decode(answer, skip_special_tokens=True)))

    if not records:
        raise ValueError("Учитель не вернул ни одного ответа.")
    (output_path / "answers.jsonl").write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records),
        encoding="utf-8",
    )
    print(f"Сохранено ответов: {len(records)} / {len(row_list)}")
    return output_path

In [19]:
distill_rows = train_data.shuffle(seed=SEED).select(
    range(min(DISTILL_SAMPLES, len(train_data)))
)
TEACHER_ANSWERS = collect_teacher_answers(
    teacher, tokenizer, distill_rows, ARTIFACTS / "teacher_answers"
)
del teacher
clean_memory()

Сохранено ответов: 384 / 384

### Реализация пайплайна обучения ученика (1 балл)

Полное дообучение Qwen3-8B на A100 80GB проходит на грани по памяти. Для решения можно включать gradient checkpointing, уменьшать batch или брать меньшую модель того же семейства.

In [20]:
class TeacherAnswerDataset(torch.utils.data.Dataset):
    """Примеры для обучения ученика на ответах учителя."""

    def __init__(self, path: str | Path) -> None:
        """

        Args:
            path: каталог с answers.jsonl из collect_teacher_answers.
        """
        with (Path(path) / "answers.jsonl").open(encoding="utf-8") as f:
            self.rows = [json.loads(line) for line in f]

    def __len__(self) -> int:
        """
            Число строк в answers.jsonl.
        """
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, list[int]]:
        """
        Args:
            index: номер примера.

        Returns:
            Словарь с `input_ids` (промпт + ответ), `attention_mask` и `labels`,
            токены промпта маскируем значением -100.
        """
        row = self.rows[index]
        input_ids = row["prompt_ids"] + row["answer_ids"]
        labels = [-100] * len(row["prompt_ids"]) + row["answer_ids"]
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }

def collate_teacher_answers(features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    """Паддим примеры справа до общей длины батча.

    Args:
        features: примеры из TeacherAnswerDataset.

    Returns:
        Тензоры `input_ids`, `attention_mask` и `labels` формы [batch, length];
        паддинг в `labels` равен -100.
    """
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    width = max(len(feature["input_ids"]) for feature in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for feature in features:
        pad = width - len(feature["input_ids"])
        batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad)
        batch["attention_mask"].append(feature["attention_mask"] + [0] * pad)
        batch["labels"].append(feature["labels"] + [-100] * pad)
    return {key: torch.tensor(value) for key, value in batch.items()}

In [21]:
def train_student(name, learning_rate):
    set_seed(SEED)
    student = AutoModelForCausalLM.from_pretrained(BASELINE_PATH, torch_dtype=torch.bfloat16,
                                                   device_map=DEVICE)
    adapter = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
                         task_type="CAUSAL_LM")
    student = get_peft_model(student, adapter)
    path = ARTIFACTS / name
    args = TrainingArguments(
        output_dir=str(path), learning_rate=learning_rate, num_train_epochs=2,
        per_device_train_batch_size=4, gradient_accumulation_steps=2,
        bf16=True, logging_steps=8, save_strategy="no", report_to="none", seed=SEED,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    trainer = Trainer(model=student, args=args,
                      train_dataset=TeacherAnswerDataset(ARTIFACTS / "teacher_answers"),
                      data_collator=collate_teacher_answers)
    trainer.train()
    trainer.save_model(path)
    config_path = path / "adapter_config.json"
    config = json.loads(config_path.read_text())
    config.update(base_model_name_or_path=BASE_MODEL_ID, revision=Path(BASELINE_PATH).name)
    save_json(config_path, config)
    tokenizer.save_pretrained(path)
    save_json(path / "training_log.json", trainer.state.log_history)
    del trainer, student
    clean_memory()
    metrics, predictions = evaluate_checkpoint(path, dev_data)
    save_dev(name, metrics, predictions)

In [22]:
train_student("sft_lr1e4", 1e-4)

Step,Training Loss


sft_lr1e4 {'quality': 0.5814609945255474, 'intent_accuracy': 0.7578125, 'slot_micro_f1': 0.4051094890510949, 'frame_exact_match': 0.3046875, 'valid_json_rate': 0.96875, 'n': 256}

### Добейтесь максимального качества ученика. Цель — не более 10% отклонения от учителя (2 балла)
Пробуйте менять гиперпараметры, число сэмплов, модели ученика (при условии размера модели ученика не более 8B).

In [23]:
train_student("sft_lr3e4", 3e-4)
candidates = ["sft_lr1e4", "sft_lr3e4"]
def quality(run):
    return json.loads((ARTIFACTS / f"{run}_dev.json").read_text())["quality"]
best = max(candidates, key=quality)
shutil.copytree(ARTIFACTS / best, ARTIFACTS / "student_sft", dirs_exist_ok=True)
for suffix in ("dev.json", "predictions.jsonl"):
    shutil.copyfile(ARTIFACTS / f"{best}_{suffix}", ARTIFACTS / f"student_{suffix}")
print(best, quality(best), "цель", 0.9 * quality("teacher"))

Step,Training Loss


sft_lr3e4 {'quality': 0.5926509296594982, 'intent_accuracy': 0.76953125, 'slot_micro_f1': 0.4157706093189964, 'frame_exact_match': 0.2890625, 'valid_json_rate': 0.97265625, 'n': 256}
sft_lr3e4 0.5926509296594982 цель 0.5205769496681417

### Выводы по обучению

Я обучил два адаптера на одних и тех же 384 ответах учителя. При learning rate 1e-4 получил Q = 0,581, при learning rate 3e-4 — 0,593. Я выбрал второй адаптер по Q на 256 примерах dev и сохранил его в `artifacts_compression_hw/student_sft`. Исходная Qwen3-8B получила на них Q = 0,278.

У учителя Q = 0,578. Результат ученика составил 102,5% от Q учителя, то есть примерно на 2,5% выше. Это возможно, потому что на dev ответы обеих моделей сравниваются с эталонной разметкой MASSIVE. Ученик специализировался на задаче и мог точнее ответить на новые запросы. В измерениях выигрыш дал Slot Micro F1: 0,416 у ученика против 0,372 у учителя. Он перевесил снижение Intent Accuracy с 0,785 до 0,770. Обе метрики входят в Q с равным весом.

Эти 102,5% получены на той же валидации, по которой я выбрал learning rate. На отдельном тесте преимущество ученика я не проверял.